In [1]:
import pandas as pd

In [2]:
consultations_df = pd.read_csv('raw/consultations_seed.csv')
product_catalog_df = pd.read_csv('raw/product_catalog.csv')
products_for_prompt_df = pd.read_csv('raw/products_for_prompt.csv')

In [3]:
consultations_df.head()

,consultation_id,age,skin_type,concerns,budget,allergies,values,experience,products_recommended,total_price,reasoning,warning_notes,full_reasoning,status
0,cons_001,22,Жирная с акне,"Акне, расширенные поры, себорегуляция",low,NaN,Баланс (качество + цена),Начинающий,"product_007, product_027, product_028, product...","3,957",COSRX Low pH (мягкий BHA для новичков) + DE_CO...,⚠️ Вводить кислоты МЕДЛЕННО (1 неделя только A...,---\n\n# ПЕРСОНАЛЬНЫЙ УХОД: ЖИРНАЯ С АКНЕ\n\n#...,True
1,cons_002,28,Комбинированная (T-зона жирная),"Жирность T-зоны, матирование, гидратация щек",medium,Propylene Glycol,Баланс,Средний,"product_018, product_015, product_035, product...","7,219","COSRX AHA/BHA/C Toner (для T-зоны контроль), 8...",⚠️ Propylene Glycol в COSRX может быть проблем...,---\n\n# ПЕРСОНАЛЬНЫЙ УХОД: КОМБИНИРОВАННАЯ (T...,True
2,cons_003,45,"Сухая, зрелая","Морщины, потеря упругости, обезвоживание",high,NaN,Люкс (инновация),Продвинутый,"product_050, product_048, product_045, product...","12,408","PAYOT Creme Riche (ночь, питание), Round Lab B...",🚨 JASMINUM OIL в PAYOT НОЧЬ ТОЛЬКО (фототоксич...,"---\n\n# ПЕРСОНАЛЬНЫЙ УХОД: СУХАЯ, ЗРЕЛАЯ КОЖА...",True
3,cons_004,19,"Чувствительная, склонна к раздражению","Чувствительность, воспаление, раздражение",low,"Salicylic Acid, Fragrance",Минимализм (только необходимое),Начинающий,"product_002, product_023, product_042, product...","5,18","Bioderma Sensibio Gel (классик для sensitive),...",❌ НЕ ДОБАВЛЯТЬ кислоты! ⚠️ Если воспаление уси...,---\n\n# ПЕРСОНАЛЬНЫЙ УХОД: ЧУВСТВИТЕЛЬНАЯ КОЖ...,True
4,cons_005,31,Жирная с постакне,"Пигментация (постакне), себум, рубцы",medium,NaN,Баланс + результативность,Средний,"product_007, product_052, product_059, product...","6,887","COSRX Low pH (очищение), ART & FACT Azeloglici...",🚨 Azeloglicine 10% МОЩНА! Вводить медленно (2 ...,---\n\n# ПЕРСОНАЛЬНЫЙ УХОД: ЖИРНАЯ КОЖА С ПОСТ...,True


In [14]:
consultations_df['allergies'] = consultations_df['allergies'].fillna('отсутствуют')

In [27]:
def build_prompt(row):
    return f"""# ДАННЫЕ КЛИЕНТА:

**Возраст:** {row['age']}
**Тип кожи:** {row['skin_type']}
**Основные проблемы:** {row['concerns']}
**Бюджет:** {row['budget']}₽
**Аллергии/чувствительность:** {row['allergies']}
**Опыт ухода:** {row['experience']}
**Ценности:** {row['values']}

---

**ТВОЯ ЗАДАЧА:**
1. Выбрать 5-10 оптимальных продуктов из каталога
2. Написать полную консультацию по шаблону (см. system prompt)
3. Объяснить для каждого продукта, ПОЧЕМУ он выбран именно для этого клиента
4. Дать точный ROUTINE и ЗАПРЕТЫ
5. ИТОГО стоимость в рамках бюджета

**НАЧИНАЙ:**
"""

consultations_df['prompt'] = consultations_df.apply(build_prompt, axis=1)

In [28]:
print(str(consultations_df.loc[:0, ['prompt']].to_string(index=False)))

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                prompt
# ДАННЫЕ КЛИЕНТА:\n\n**Возраст:** 22\n**Тип кожи:** Жирная с акне\n**Основные проблемы:** Акне, расширенные поры, себорегуляция\n**Бюджет:** low₽\n**Аллергии/чувствительность:** отсутствуют\n**Опыт ухода:** Начинающий\n**Ценности:** Баланс (качество + цена)\n\n---\n\n**ТВОЯ ЗАДАЧА:**\n1. Выбрать 5-10 оптимальных продуктов из каталога\n2. Написать полную консультацию по шаблону (см. system prompt)\n3. Объяснить для каждого продукт

In [34]:
df = consultations_df[['consultation_id', 'prompt', 'full_reasoning']].rename(columns={'consultation_id': 'id', 'full_reasoning': 'response'})

In [4]:
product_catalog_df.head()

,product_id,name,brand,price_rub,segment,category,link,ingredients_raw,main_actives,functional_category,pH,allergens,vegan,cruelty_free,format,volume_ml,price_per_ml,issues
0,Уникальный ID продукта,Название продукта,Бренд,Цена в рублях (актуальная на дату),бюджетное/мидл/люкс,Категория продукта,Ссылка на Ozon (если есть),Полный список ингредиентов (копипаст),Главные активные ингредиенты + концентрация,Функция (может быть несколько),Кислотность продукта (если известен),Известные раздражители,Веган ли,Крuelty-free ли,Формат выпуска,Объем в мл,Цена за 1мл (вычисляется),Если есть проблемы
1,product_001,Мягкая пенка с увлажняющим мультикомплексом,Pleyana,743,бюджет,Очищающий гель/пенка,https://www.pleyana.com/catalog/profy/myagkaya...,"Aqua, Disodium Cocoyl Glutamate, Coco-Glucosid...","Disodium Cocoyl Glutamate (12-15%, ПАВ растите...","Мягкое очищение, восстановление барьера, гидра...",5.5 (хороший pH!),"Dichlorobenzyl Alcohol (консервант, более мягк...",Да,Да,Пенка в флаконе,75,"9,906666667",✅ Хороший pH 5.5; ✅ Мягкий ПАВ (Disodium Cocoy...
2,product_002,Очищающий гель для лица Sensibio,Bioderma,865,бюджет,Очищающий гель,https://goldapple.ru/19000041345-sensibio,"Aqua/water/eau, Sodium Cocoamphoacetate, Propa...",Пребиотический комплекс (Mannitol + Xylitol + ...,"Деликатное очищение, восстановление микробиома...",5.0-5.5,Mannitol/Xylitol (редко) — отсутствуют типичны...,Да,Неизвестно,Гель,100,"8,65",✅ КАЧЕСТВЕННЫЙ; ✅ Нет консервантов-раздражител...
3,product_003,Гель для умывания Sebium,Bioderma,1458,мидл,Очищающий гель,goldapple.ru/89270300017-sebium,"Aqua (water/eau), Sodium cocoamphoacetate, Sod...","Лактическая кислота (~1-2%, мягкий AHA), Цинк ...","Мягкое очищение, себорегуляция, пилинг (мягкий...",3.5-4.0,Sodium laureth sulfate (раздражитель для sensi...,Да,Да,Гель,200,"7,29",🚨 CRITICAL: Sodium laureth sulfate (SLS) — АГР...
4,product_004,Hyaluronic Micellar Water,Aravia Laboratories,505,бюджет,Мицеллярная вода,NaN,"Aqua, Glycerin, Poloxamer 184, PEG-40 Hydrogen...",NaN,"Мягкое очищение, гидратация, успокаивание",46148,"Benzyl Alcohol (консервант-раздражитель), отсу...",Да,Да,Жидкость,520,"0,9711538462",✅ ОЧЕНЬ ДЕШЕВО (0.97₽/мл) — лучший ratio. ✅ Пр...


In [ ]:
def build_prompt(row):
    return f"""# ДАННЫЕ КЛИЕНТА:

**Возраст:** {row['age']}
**Тип кожи:** {row['skin_type']}
**Основные проблемы:** {row['concerns']}
**Бюджет:** {row['budget']}₽
**Аллергии/чувствительность:** {row['allergies']}
**Опыт ухода:** {row['experience']}
**Ценности:** {row['values']}

---

**ТВОЯ ЗАДАЧА:**
1. Выбрать 5-10 оптимальных продуктов из каталога
2. Написать полную консультацию по шаблону (см. system prompt)
3. Объяснить для каждого продукта, ПОЧЕМУ он выбран именно для этого клиента
4. Дать точный ROUTINE и ЗАПРЕТЫ
5. ИТОГО стоимость в рамках бюджета

**НАЧИНАЙ:**
"""

consultations_df['prompt'] = consultations_df.apply(build_prompt, axis=1)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70 entries, 0 to 69
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   product_id           70 non-null     object
 1   name                 70 non-null     object
 2   brand                70 non-null     object
 3   price_rub            70 non-null     object
 4   segment              70 non-null     object
 5   category             70 non-null     object
 6   link                 69 non-null     object
 7   ingredients_raw      70 non-null     object
 8   main_actives         69 non-null     object
 9   functional_category  70 non-null     object
 10  pH                   70 non-null     object
 11  allergens            70 non-null     object
 12  vegan                70 non-null     object
 13  cruelty_free         70 non-null     object
 14  format               70 non-null     object
 15  volume_ml            70 non-null     object
 16  price_per_

In [6]:
product_catalog_df.fillna('Неизвестно', inplace=True)

In [8]:
product_catalog_df = product_catalog_df[cols]

In [9]:
product_catalog_df.iloc[1:, :].to_json('preprocessed/product_catalog.json', orient='records', force_ascii=False, indent=2)

In [10]:
import json

with open('preprocessed/product_catalog.json', 'r', encoding='utf-8') as f:
    product_catalog = json.load(f)

product_catalog_str = json.dumps(product_catalog, ensure_ascii=False, indent=2)

In [11]:
system_prompt = f"""
Ты — KOYASH AI, профессиональный консультант по косметике. Твоя задача:
1. ПРОЧИТАТЬ профиль клиента
2. ВЫБРАТЬ оптимальные продукты из каталога (5-10 штук)
3. НАПИСАТЬ полную, развёрнутую консультацию (как реальный косметолог)

═══════════════════════════════════════════════════════════════════════════

ЖЁСТКИЕ ПРАВИЛА:

1. КАТАЛОГ — СВЯЩЕННОЕ (рекомендуй ТОЛЬКО отсюда, не выдумывай):
{product_catalog_str}

2. КРИТЕРИИ ВЫБОРА:
   ✅ Тип кожи ТОЧНО совпадает
   ✅ Решает основные проблемы клиента
   ✅ В рамках бюджета (±10% max)
   ✅ НЕ содержит аллергены из списка
   ✅ Совместимы между собой (проверь INCI-комбинации)
   ✅ 5-10 продуктов (не меньше, не больше)

3. ПРИОРИТЕТ ПРОДУКТОВ (важность):
   1️⃣ ОЧИЩЕНИЕ (обязательно)
   2️⃣ SPF (если есть кислоты, обязателен!)
   3️⃣ РЕШЕНИЕ ОСНОВНОЙ ПРОБЛЕМЫ (акне/морщины/пигментация)
   4️⃣ ГИДРАТАЦИЯ/ВОССТАНОВЛЕНИЕ БАРЬЕРА
   5️⃣ ДОПОЛНИТЕЛЬНЫЕ (маски, сыворотки, бонусы)

4. АЛЛЕРГИИ — ТАБУ:
   Если аллерген есть в продукте → ИСКЛЮЧИ, даже если он идеален
   Если нет аналога → честно скажи клиенту

5. БЮДЖЕТ:
   Если сумма товаров > бюджет → убери менее критичные
   Если есть запас (>20%) → добавь дополнительный бонус

6. НАУЧНОЕ ОБОСНОВАНИЕ:
   Для каждого продукта объясни МЕХАНИЗМ (как работает компонент)
   Не маркетинг, а реальные INCI и их свойства

═══════════════════════════════════════════════════════════════════════════

СТРУКТУРА ОТВЕТА (ТОЧНО СЛЕДУЙ):

# ПЕРСОНАЛЬНЫЙ УХОД: [ТИП КОЖИ]

## РАЗБОР ПРОДУКТОВ

### 1. [БРЕНД] [НАЗВАНИЕ] — [КАТЕГОРИЯ]
**Цена:** Х₽
**Назначение:** [2 предложения: зачем именно для этого клиента, какой ингредиент решает какую проблему]
**Активные компоненты:**
- **[Компонент 1]** — [функция]
- **[Компонент 2]** — [функция]
**pH:** [если есть]
**Применение:**
- Утро/Вечер: [когда]
- Частота: [сколько раз в неделю]
- Техника: [как наносить]
**Правило:** [Главное правило применения]

[...повтори для всех продуктов...]

## ROUTINE

**Утро:**
1. [Продукт] — [зачем]
2. [Продукт] — [зачем]

**Вечер (базовый):**
1. [Продукт] — [зачем]
2. [Продукт] — [зачем]

**Вечер (пилинг-дни или специальные):**
1. [Продукт] — [когда]

## ЗАПРЕЩЁННЫЕ КОМБИНАЦИИ
❌ [Продукт А] + [Продукт Б] в один день — [почему]
⚠️ [Условие] — [объяснение]

## СТОП-СИГНАЛЫ
Прекрати если:
- [симптом 1]
- [симптом 2]
- [симптом 3]

## ТАЙМЛАЙН РЕЗУЛЬТАТОВ
- **Неделя 1-2:** [ожидания]
- **Неделя 3-4:** [результаты]
- **Месяц 1-3:** [долгосрочный результат]

**ИТОГО: Х₽ (в рамках бюджета [Y]₽)**

**Желаю вам здоровой кожи!** ✨

═══════════════════════════════════════════════════════════════════════════

СТИЛЬ:
- Структурированный, научный, без воды
- Честный, как подруга, которая разбирается
- Практичный, с точными инструкциями
- 800-1200 слов (ни меньше, ни больше)

КАТЕГОРИЧЕСКИ ЗАПРЕЩЕНО:
❌ Придумывать продукты вне каталога
❌ Рекомендовать при наличии аллергена
❌ Писать маркетинговый текст (только научный)
❌ Больше 10 продуктов или меньше 5
❌ Превышать бюджет >10%
❌ Забывать про ROUTINE или ЗАПРЕТЫ
"""

In [12]:
with open('preprocessed/system_prompt.txt', 'w', encoding='utf-8') as f:
    f.write(system_prompt)

In [ ]:
import ollama

user_prompt = """
Профиль пользователя:
- Тип кожи: жирная, склонная к акне
- Проблемы: расширенные поры, постакне
- Бюджет: 5000₽
- Аллергии: отдушки, спирт
- Опыт: новичок
"""

response = ollama.chat(
    model="qwen2.5:32b",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt},
    ],
    options={"temperature": 0.3},
)

print(response["message"]["content"])

РЕКОМЕНДОВАННАЯ КОСМЕТИЧКА:

1. **CLEANSER**: Gentle Foam Cleanser (Cetaphil, 500₽)
   Зачем: Мягко очищает кожу без раздражения, подходит для чувствительной кожи.

2. **TONER**: Tea Tree Toner (The Ordinary, 450₽)
   Зачем: Чайное дерево помогает бороться с акне и уменьшить воспаление.

3. **TREATMENT**: Niacinamide 10% + Zinc 1% (The Ordinary, 600₽)
   Зачем: Ниацинамид помогает контролировать выработку себума, уменьшать покраснения и сужать поры.

4. **MOISTURIZER**: Oil-Free Moisturizer (Neutrogena, 750₽)
   Зачем: Увлажняет кожу без добавления жира, поддерживая баланс кожи.

5. **SUNSCREEN**: Dual Barrier Watery Sun Cream SPF 50+ PA++++ (Celimax, 1112₽)
   Зачем: Минеральный солнцезащитный крем для чувствительной кожи, помогает предотвратить повреждение от УФ-лучей.

6. **EXFOLIANT**: Lactic Acid Toning Solution 5% (The Ordinary, 400₽)
   Зачем: Лактатная кислота мягко эксфолиирует кожу, помогая уменьшить постакне и сужать поры.

7. **SPOT TREATMENT**: Salicylic Acid 2% Masque (Th

In [10]:
df = pd.DataFrame({'id':[], 'prompt':[], 'response':[]})

In [35]:
df.to_csv('preprocessed/metrics_eval.csv', index=False, encoding='utf-8')